# Enedis — entrenament Transformer v3.1 (Kaggle)

Arquitectura actual (`lstm/src/config.py`): **D_MODEL=192**, **N_HEADS=8**, **4+4 capes**, **D_FF=768**.

Un sol procés de 80 èpoques. Després de **cada** època es guarda `latest.pt` a `/kaggle/working/DataChallenge_checkpoints/`.
Si Kaggle s'atura, una sessió nova amb `RESUME=True` continua a l'època següent.

**Codi:** clone del repositori privat amb el secret `GITHUB_TOKEN` (Add-ons → Secrets).
**Dades:** CSVs no van al git; es recullen d'un dataset Kaggle (`X_train.csv` a `/kaggle/input/...`).

In [ ]:
# ── Configuració (edita només això) ──────────────────────────────────────────
RESUME = False

# Si RESUME=True:
#   None → /kaggle/working/DataChallenge_checkpoints/latest.pt
RESUME_CHECKPOINT = None

GITHUB_REPO = "https://github.com/PolTibau/DataChallenge.git"
GITHUB_BRANCH = "main"
CLONE_DIR = "/kaggle/working/DataChallenge"
KAGGLE_CKPT_DIR = "/kaggle/working/DataChallenge_checkpoints"

In [ ]:
# Clone del repo privat (secret GITHUB_TOKEN a Kaggle)
import os
import subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
clone_url = GITHUB_REPO.replace("https://", f"https://oauth2:{token}@")
clone_dir = Path(CLONE_DIR)

if (clone_dir / "lstm" / "main_train_combined.py").exists():
    print(f"Repo ja present: {clone_dir}")
    subprocess.run(
        ["git", "-C", str(clone_dir), "fetch", "origin", GITHUB_BRANCH],
        check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )
    subprocess.run(
        ["git", "-C", str(clone_dir), "checkout", GITHUB_BRANCH], check=True,
    )
    subprocess.run(
        ["git", "-C", str(clone_dir), "pull", "--ff-only", "origin", GITHUB_BRANCH],
        check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )
else:
    if clone_dir.exists():
        raise FileExistsError(f"{clone_dir} existeix però no és el repo")
    subprocess.run(
        ["git", "clone", "--branch", GITHUB_BRANCH, "--depth", "1",
         clone_url, str(clone_dir)],
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )

print("Clone OK:", clone_dir)
print("HEAD:")
subprocess.run(["git", "-C", str(clone_dir), "log", "-1", "--oneline"])

In [ ]:
# Dades: el gitignore exclou els CSV. Enllaça un dataset Kaggle amb X_train.csv.
import os
import shutil
from pathlib import Path

data_dest = Path(CLONE_DIR) / "data"
data_dest.mkdir(parents=True, exist_ok=True)

needed = ["X_train.csv", "y_train.csv", "X_test.csv"]
src_dir = None
input_root = Path("/kaggle/input")
if input_root.exists():
    for hit in input_root.rglob("X_train.csv"):
        src_dir = hit.parent
        break

if src_dir is None:
    missing = [n for n in needed if not (data_dest / n).exists()]
    if missing:
        raise FileNotFoundError(
            "No s'ha trobat X_train.csv a /kaggle/input. Afegeix el dataset de dades.\n"
            f"Falten a {data_dest}: {missing}"
        )
    print("CSVs ja presents a", data_dest)
else:
    print("Dades origen:", src_dir)
    for name in needed:
        src, dst = src_dir / name, data_dest / name
        if not src.exists():
            raise FileNotFoundError(src)
        if dst.exists() or dst.is_symlink():
            print(f"  skip {name} (ja hi és)")
            continue
        try:
            os.symlink(src, dst)
            print(f"  symlink {name}")
        except OSError:
            shutil.copy2(src, dst)
            print(f"  copy {name}")

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path(CLONE_DIR)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("cwd :", Path.cwd())
print("ROOT:", ROOT)

In [ ]:
import lstm.main_train_combined as train_script
from lstm.src import config as cfg

train_script.RESUME = RESUME
train_script.RESUME_FROM = RESUME_CHECKPOINT

print("RESUME         :", train_script.RESUME)
print("RESUME_FROM    :", train_script.RESUME_FROM)
print("ckpt_dir       :", train_script.resolve_ckpt_dir())
print(
    f"model          : D_MODEL={cfg.D_MODEL}  N_HEADS={cfg.N_HEADS}  "
    f"enc/dec={cfg.N_ENC_LAYERS}+{cfg.N_DEC_LAYERS}  D_FF={cfg.D_FF}  "
    f"N_EPOCHS={cfg.N_EPOCHS}"
)

In [ ]:
# Entrenament únic (80 èpoques, latest.pt després de cada època)
train_script.main()

In [ ]:
from pathlib import Path

ckpt_dir = Path("/kaggle/working/DataChallenge_checkpoints")
if not ckpt_dir.exists():
    ckpt_dir = train_script.resolve_ckpt_dir()

print("Checkpoints a", ckpt_dir)
if ckpt_dir.exists():
    for p in sorted(ckpt_dir.iterdir()):
        print(f"  {p.name:24s}  {p.stat().st_size/1e6:.2f} MB")
else:
    print("  (directori inexistent)")

In [ ]:
from pathlib import Path
import json

ckpt_dir = Path("/kaggle/working/DataChallenge_checkpoints")
if not ckpt_dir.exists():
    ckpt_dir = train_script.resolve_ckpt_dir()

status_path = ckpt_dir / "training_status.json"
status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.exists() else {}

latest = ckpt_dir / "latest.pt"
best_full = ckpt_dir / "best.pt"
best_weights = Path(status.get("best_weights", train_script.COMBINED_CKPT))

print("Training finished")
print(f"Last checkpoint: {latest if latest.exists() else status.get('latest')}")
print(f"Best checkpoint/model: {best_weights}")
print(f"Last completed epoch: {status.get('last_completed_epoch', '?')}")
if best_full.exists():
    print(f"Best full checkpoint (resume): {best_full}")
print("\nPer continuar: RESUME=True i torna a executar (el token clona el repo de nou;")
print("els checkpoints queden a /kaggle/working si deses la versió o els afegeixes com a output).")